# Spatial Feature Experiment: Distance to Beach vs. Distance to City Center

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`,
so the earlier model comparison stays as-is. Here we isolate the effect of two
engineered spatial features on top of raw `latitude`/`longitude`:

- **Baseline**: raw `latitude` + `longitude` only (no engineered distances)
- **+ dist_to_beach**: distance to Ipanema Beach
- **+ dist_to_beach + dist_to_city_center**: distance to Ipanema Beach AND distance to Centro (Rio's city center)

We train XGBoost on each variant (same split, same target) and compare R^2/MAE/RMSE.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target, clean_numerical_features, clean_price_column, remove_price_outliers_dynamic
from src.evaluate import print_metrics_log_target
from src.features.selection import select_important_features
from src.features.spatial import add_beach_distance, add_city_center_distance
from src.features.text import add_amenity_count, add_description_word_count
from src.models.tree_models import train_xgboost


## Step 1: Clean + select base features (no spatial distances yet)

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)

df = clean_numerical_features(df)
df = select_important_features(df)
df = clean_price_column(df)
df = remove_price_outliers_dynamic(df)
df = add_amenity_count(df)
df = add_description_word_count(df)

df_base = df.copy()
df_base.shape


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


(39417, 26)

## Step 2: Build the three feature variants

In [3]:
df_variant_a = df_base.copy()  # baseline: raw lat/lon only

df_variant_b = add_beach_distance(df_base.copy())  # + dist_to_beach

df_variant_c = add_beach_distance(df_base.copy())
df_variant_c = add_city_center_distance(df_variant_c)  # + dist_to_beach + dist_to_city_center

variants = {
    "A: lat/lon only": df_variant_a,
    "B: + dist_to_beach": df_variant_b,
    "C: + dist_to_beach + dist_to_city_center": df_variant_c,
}


## Step 3: Train XGBoost on each variant and compare

In [4]:
def evaluate_variant(name, variant_df):
    X = variant_df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number])
    y_log = np.log1p(variant_df[TARGET_COLUMN])

    X_train, X_test, y_train_log, y_test_log = train_test_split(
        X, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = train_xgboost(X_train, y_train_log)
    preds_log = model.predict(X_test)
    metrics = print_metrics_log_target(name, y_test_log, preds_log)
    metrics["n_features"] = X.shape[1]
    return metrics


results = {name: evaluate_variant(name, variant_df) for name, variant_df in variants.items()}


A: lat/lon only                | R^2 Score: 61.74% | MAE: $255.68 | RMSE: $513.85


B: + dist_to_beach             | R^2 Score: 61.94% | MAE: $255.31 | RMSE: $514.42


C: + dist_to_beach + dist_to_city_center | R^2 Score: 61.79% | MAE: $256.26 | RMSE: $514.95


## Step 4: Summary table

In [5]:
comparison = pd.DataFrame(results).T[["n_features", "mae", "rmse", "r2"]]
comparison


,n_features,mae,rmse,r2
A: lat/lon only,16.0,255.680693,513.852779,0.617386
B: + dist_to_beach,17.0,255.313469,514.418359,0.619391
C: + dist_to_beach + dist_to_city_center,18.0,256.258898,514.950075,0.617932
